# Distribuzioni di carica per canale

Si cambia solo la cella **CONFIGURAZIONE** qui sotto, poi *Run All*. Come funziona: in fondo.

In [ ]:
# ===================== CONFIGURAZIONE =====================
# --- run ---
RUN      = 158                 # numero, range(150, 159), lista [150, 158], nome, glob "LNGS__54-0V_015?_*"
PREFISSO = "LNGS__54-0V_"      # per le run date per numero (la numerazione riparte a ogni prefisso)
CARTELLA = None                # None: data/LNGS sul PC, ~/nmv-data/WC/LNGS sul Mac

# --- canali ---
CANALI     = [16, 17, 18, 20, 21]
RIVELATORE = {16: "SiPM", 17: "SiPM", 18: "SiPM", 20: "PMT", 21: "PMT"}

# --- cancello automatico, per rivelatore ---
# soglia: frazione del picco della forma media che delimita l'impulso
# prima_ns / dopo_ns: margini prima e dopo gli attraversamenti della soglia
# fine_traccia: se non None, il cancello arriva in fondo alla traccia meno questi campioni
CANCELLO_AUTO = {
    "SiPM": {"soglia": 0.10, "prima_ns": 60, "dopo_ns": 20, "fine_traccia": 20},
    "PMT":  {"soglia": 0.10, "prima_ns": 10, "dopo_ns": 20, "fine_traccia": None},
}
SOGLIA_FORMA = {"SiPM": 100, "PMT": 73}   # conteggi sopra la base: eventi per forma media e tempo
                                          # del massimo (PMT: mezzo PE, 1 PE ~ 146 conteggi)

# --- piedistallo, per rivelatore ---
# inizio_ns: dove comincia la finestra; distacco_ns: quanto prima del cancello finisce
# metodo: "evento" (media della finestra in ogni evento) o "blocco" (mediana di quelle medie
#         su EVENTI_PER_BLOCCO eventi consecutivi)
PIEDISTALLO = {
    "SiPM": {"inizio_ns": 10, "distacco_ns": 20, "metodo": "blocco"},
    "PMT":  {"inizio_ns": 10, "distacco_ns": 20, "metodo": "evento"},
}
EVENTI_PER_BLOCCO = 5000       # eventi consecutivi per il piedistallo "blocco" (~90 s a 57 Hz)

# --- a mano, per canale: vincono sugli automatici ---
CANCELLO     = {}              # ns, es. {20: (240, 500), 21: (240, 500)}
FINESTRA_PED = {}              # ns, es. {c: (0, 150) for c in (16, 17, 18)}
VERSO        = {}              # +1 / -1 se il DCOffset nel log manca o sbaglia, es. {20: -1}
# ricetta 0011/0013: CANCELLO = {c: (240, 500) for c in CANALI}; FINESTRA_PED = {c: (0, 150) for c in CANALI}

# --- selezione degli eventi (data card) ---
# variabili per canale, es. ch16: A16 ampiezza nel cancello [mV], Q16 carica [PE] (serve PE_ADCNS),
#   C16 carica [ADC*ns], T16 istante del massimo nel cancello [ns], S16 saturo (vero/falso)
# operatori: > < >= <= == !=   and or not   ( )   + - * /   abs() min() max()
# contatori: almeno(n, ...), al_piu(n, ...), esattamente(n, ...) = quante condizioni sono vere
# le CONDIZIONI hanno un nome, da usare nella SELEZIONE e nelle condizioni dopo; SELEZIONE = "" -> tutti
CONDIZIONI = {
    "pannelli2":    "almeno(2, A16 > 50, A17 > 50, A18 > 50)",
    "pmt_entrambi": "Q20 > 0.5 and Q21 > 0.5",
    "pmt_uno_solo": "esattamente(1, Q20 > 0.5, Q21 > 0.5)",
}
SELEZIONE = ""                 # es. "pannelli2 and pmt_entrambi", "pmt_uno_solo and not pannelli2"
PE_ADCNS  = {20: 1719, 21: 1709}   # carica di un PE [ADC*ns]: PMT invertiti, scan del 30/10; SiPM non misurata

# --- lettura ---
EVENTI_PER_FINESTRE   = 5000   # eventi della prima run per forma media e cancelli
CAMPIONI_FINALI_ESCLUSI = 20   # gli ultimi campioni hanno punte: fuori dalla ricerca del massimo
MAX_EVENTI_PER_RUN    = None   # None = tutti; un numero per un'occhiata veloce
SCARICHI_IN_PARALLELO = 4      # run scaricate insieme dal bucket: cambia solo la velocita'
COPIA_LOCALE = "~/nmv-copia/WC/LNGS"   # le run scaricate dal bucket restano qui; None = non tenerle
SPAZIO_LIBERO_MIN_GB = 50      # sotto, una run nuova si legge senza tenerne la copia

# --- grafici ---
XLIM      = {}                 # ADC*ns per canale, es. {20: (-2000, 40000)}; vuoto = automatico
N_BIN     = 200
MAX_CURVE = 10                 # curve nel confronto fra run; oltre, run consecutive sommate

# --- uscite ---
USCITA = None                  # grafici e .json; None: measurements/
DATI   = None                  # cariche evento per evento (.npz); None: data/carica/

In [ ]:
## === Preparazione: import, cartelle, controllo della configurazione ===
import concurrent.futures
import contextlib
import glob
import gzip
import json
import os
import re
import sys
import shutil
import tempfile
import threading
import time
import zlib

import numpy as np
import matplotlib.pyplot as plt
import h5py

# La radice del repository si cerca risalendo dalla cartella del notebook (VS
# Code la dice in __vsc_ipynb_file__) o, altrimenti, da quella del kernel.
_partenza = os.path.dirname(globals().get("__vsc_ipynb_file__", "")) or os.getcwd()
ROOT = os.path.abspath(_partenza)
while not os.path.exists(os.path.join(ROOT, "tools", "daqio.py")):
    if os.path.dirname(ROOT) == ROOT:
        raise RuntimeError(
            "Non trovo il repository daq-standalone risalendo da %s. Il kernel deve girare sulla "
            "macchina del notebook: sul Mac 'Python (jupyter-global)', sul PC DAQ jupyter_env. Un "
            "kernel di un altro server (per esempio del CNAF) non lo vede." % _partenza)
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, os.path.join(ROOT, "tools"))

import daqio          # header dei file
import efficienza     # verso degli impulsi dal log della run, nome della run

def risolvi_cartelle():
    """Le cartelle lasciate a None nella configurazione. Si richiama dalla cella
    della selezione: rieseguendo la configurazione per cambiare una selezione
    tornano a None."""
    global CARTELLA, USCITA, DATI, CACHE, COPIA
    COPIA = os.path.expanduser(COPIA_LOCALE) if COPIA_LOCALE else None
    if CARTELLA is None:
        CARTELLA = next((d for d in (os.path.join(ROOT, "data", "LNGS"),
                                     os.path.expanduser("~/nmv-data/WC/LNGS")) if os.path.isdir(d)),
                        os.path.expanduser("~/nmv-data/WC/LNGS"))
    USCITA = USCITA or os.path.join(ROOT, "measurements")
    DATI = DATI or os.path.join(ROOT, "data", "carica")
    CACHE = os.path.join(DATI, "run")     # le run gia' lette


risolvi_cartelle()

# Un errore nella configurazione si dice qui, non dopo minuti di lettura.
for _c in CANALI:
    _r = RIVELATORE.get(_c)
    if _r not in CANCELLO_AUTO or _r not in PIEDISTALLO or _r not in SOGLIA_FORMA:
        raise ValueError("ch%d: rivelatore %r senza regola in CANCELLO_AUTO, PIEDISTALLO o "
                         "SOGLIA_FORMA" % (_c, _r))
    if PIEDISTALLO[_r]["metodo"] not in ("evento", "blocco"):
        raise ValueError("PIEDISTALLO[%r]['metodo'] deve essere 'evento' o 'blocco'" % _r)
print("dati da %s; uscite in %s e %s" % (CARTELLA, USCITA, DATI))
if COPIA and os.path.isdir(COPIA):
    # Pezzi di copie interrotte da un kernel fermato a meta': nessuno li finira'
    for _p in glob.glob(os.path.join(COPIA, "*.part-*")):
        if time.time() - os.path.getmtime(_p) > 6 * 3600:
            os.unlink(_p)
    _g = glob.glob(os.path.join(COPIA, "*.h5.gz"))
    print("copia locale delle run del bucket: %s, %d run, %.1f GB; %.0f GB liberi sul disco"
          % (COPIA, len(_g), sum(map(os.path.getsize, _g)) / 1e9,
             shutil.disk_usage(COPIA).free / 1e9))

In [ ]:
## === Funzioni ===
TTT_NS = 17.07       # passo del Trigger Time Tag, misurato il 2026-10-23 (CLAUDE.md)
TTT_GIRO = 2 ** 31   # contatore a 31 bit: un giro ogni ~36.7 s
FONDO_SCALA = 4095   # 12 bit
VERSIONE_CACHE = 2   # da cambiare se cambia quello che va in cache (2: massimo e minimo nel cancello)
MV_PER_CONTEGGIO = 1000.0 / 4096   # 1 Vpp su 12 bit
# Il bucket del cloud montato sul Mac: li' un .h5 non e' mai una run buona
# (archivia.py manda solo .h5.gz), ma una copia presa a run in corso (0054,
# 0055) o di una run chiusa male (0005).
REMOTO = os.path.realpath(os.path.expanduser("~/nmv-data"))
LOCALI = {}          # run gia' scaricate e non ancora lette: {file: (locale, temporaneo)}
MIN_CAMPIONI_PED = 20   # sotto, una finestra del piedistallo non e' una stima


def _dati(percorsi):
    """Un file di dati per run. Col .h5 e il .h5.gz insieme la run e' appena
    stata compressa e il .h5 sta per sparire: vale il .gz, che archivia.py
    rinomina col suo nome solo quando e' completo e verificato."""
    per_run = {}
    for p in percorsi:
        if p.endswith((".h5", ".h5.gz")):
            n = efficienza.nome_run(p)
            if n not in per_run or p.endswith(".gz"):
                per_run[n] = p
    return [per_run[n] for n in sorted(per_run)]


def trova_file(scelta):
    """I file di dati della scelta RUN, nell'ordine in cui sono dati. Col
    bucket non montato si cercano nella copia locale: le run gia' scaricate
    si analizzano anche senza rete."""
    dove = CARTELLA
    if not os.path.isdir(CARTELLA):
        if (COPIA and os.path.isdir(COPIA)
                and os.path.realpath(CARTELLA).startswith(REMOTO + os.sep)):
            print("Il bucket non e' montato (manca %s): cerco le run nella copia locale %s"
                  % (CARTELLA, COPIA))
            dove = COPIA
        else:
            raise FileNotFoundError(
                "Non c'e' la cartella dei dati %s. Sul Mac le run si leggono dal bucket montato in "
                "~/nmv-data (rclone, remoto cygno-ro): e' montato? Sul PC DAQ stanno in data/LNGS."
                % CARTELLA)
    if isinstance(scelta, (str, int, np.integer)):
        scelta = [scelta]
    fuori = []
    for s in scelta:
        if isinstance(s, (int, np.integer)):
            trovati = _dati(glob.glob(os.path.join(dove, "%s%04d_*" % (PREFISSO, s))))
            if len(trovati) > 1:
                raise ValueError("Run %d con prefisso %r: piu' file (%s). Darla per nome."
                                 % (s, PREFISSO, ", ".join(map(os.path.basename, trovati))))
        elif os.path.isfile(s):
            trovati = [s]
        else:
            trovati = _dati(glob.glob(os.path.join(dove, s))
                            + glob.glob(os.path.join(dove, s + ".h5*")))
        if not trovati:
            raise FileNotFoundError("Nessun file di dati per %r in %s" % (s, dove))
        fuori += [p for p in trovati if p not in fuori]
    return fuori


def etichette(nomi):
    """La parte dei nomi che cambia da una run all'altra, tagliata ai '_':
    '0150', '0151'... o '54-0V_0158', '54-1V_0000' se cambia il prefisso.
    Ritorna (prefisso comune, etichette, suffisso comune)."""
    if len(nomi) == 1:
        return "", list(nomi), ""
    p = os.path.commonprefix(nomi)
    p = p[:p.rfind("_") + 1]
    s = os.path.commonprefix([n[::-1] for n in nomi])[::-1]
    s = s[s.find("_"):] if "_" in s else ""
    return p, [n[len(p):len(n) - len(s)] for n in nomi], s


def leggi_log(path):
    """Dal log della run: verso per canale, durata, eventi, collegamento perso,
    correzione DRS4. None se il log non c'e' (un .h5.gz copiato da solo)."""
    log = re.sub(r"\.h5(\.gz)?$", ".log", path)
    try:
        testo = open(log, errors="replace").read()
    except OSError:
        copia = _copia_di(log)       # quello copiato insieme ai dati, se il bucket non risponde
        try:
            testo = open(copia, errors="replace").read()
        except (OSError, TypeError):
            return None
        log = copia
    t = re.search(r"Acquisition time: ([0-9.]+)", testo)
    n = re.search(r"Events written to file: (\d+)", testo)
    return {"versi": efficienza.polarita_dal_log(log),
            # senza riepilogo la run e' in corso, o e' morta prima di scriverlo
            "durata_s": float(t.group(1)) if t else None,
            "eventi": int(n.group(1)) if n else None,
            "collegamento_perso": "giving up" in testo,
            "drs4": ("on" if "DRS4 corrections ENABLED" in testo else
                     "off" if re.search(r"DRS4 corrections (DISABLED|disabled)", testo) else "?")}


ORIGINE = {}         # da dove e' stata letta ogni run: copia locale, bucket, disco


def _copia_di(path):
    """Il posto nella copia locale di un file del bucket; None se non e' del
    bucket o la copia non si usa. Sul cloud un file e' il suo nome, senza
    cartella, quindi la copia puo' essere piatta."""
    if not COPIA or not os.path.realpath(path).startswith(REMOTO + os.sep):
        return None
    return os.path.join(COPIA, os.path.basename(path))


def _copia_buona(path, copia):
    """La copia c'e' ed e' grande come il file del bucket: una run rimandata
    sul cloud (la 0055 completa, per esempio) ha un'altra dimensione e si
    riscarica. Col bucket che non risponde vale la copia."""
    if not os.path.exists(copia):
        return False
    try:
        return os.path.getsize(copia) == os.path.getsize(path)
    except OSError:
        return True


def _travasa(da, a, ferma, apri_da=open):
    with apri_da(da, "rb") as dentro, open(a, "wb") as fuori:
        while True:
            if ferma is not None and ferma.is_set():
                raise InterruptedError("lettura interrotta")
            pezzo = dentro.read(16 << 20)
            if not pezzo:
                break
            fuori.write(pezzo)


def _copia_dal_bucket(path, copia, ferma):
    """Il .gz del bucket nella copia locale, col suo nome solo quando e'
    arrivato tutto: una copia a meta' non deve mai sembrare una run. Il .log
    viene con lui, per poter analizzare la run anche senza rete."""
    os.makedirs(COPIA, exist_ok=True)
    parte = "%s.part-%d-%d" % (copia, os.getpid(), threading.get_ident())
    try:
        _travasa(path, parte, ferma)
        if os.path.getsize(parte) != os.path.getsize(path):
            raise OSError("copia di %s incompleta" % os.path.basename(path))
        os.replace(parte, copia)
    finally:
        if os.path.exists(parte):
            os.unlink(parte)
    log = re.sub(r"\.h5\.gz$", ".log", path)
    with contextlib.suppress(OSError):
        shutil.copyfile(log, re.sub(r"\.h5\.gz$", ".log", copia))


_poco_spazio = []


def _spazio_per(path):
    libero = shutil.disk_usage(COPIA if os.path.isdir(COPIA) else os.path.expanduser("~")).free
    if libero - os.path.getsize(path) > SPAZIO_LIBERO_MIN_GB * 1e9:
        return True
    if not _poco_spazio:
        _poco_spazio.append(1)
        print("ATTENZIONE: %.0f GB liberi, meno di SPAZIO_LIBERO_MIN_GB = %d: le run nuove si "
              "leggono dal bucket senza tenerne la copia" % (libero / 1e9, SPAZIO_LIBERO_MIN_GB))
    return False


def scarica(path, ferma=None):
    """(file da aprire, temporaneo da cancellare dopo, o None). Un .gz si
    decomprime in un file temporaneo: l'HDF5 salta da un punto all'altro del
    file, e dal bucket montato ogni salto sarebbe una richiesta in rete, quindi
    si legge tutto una volta sola, di fila. Una run del bucket prima si copia
    in COPIA_LOCALE, e la volta dopo si legge da li': lo scaricamento costa
    minuti, la decompressione pochi secondi. `ferma` interrompe a meta'."""
    if not path.endswith(".gz"):
        if os.path.realpath(path).startswith(REMOTO + os.sep):
            raise ValueError("sul cloud c'e' solo il .h5, una copia presa a run in corso o di una "
                             "run chiusa male: la run completa e' il .h5.gz")
        ORIGINE[path] = "dal disco"
        return path, None
    sorgente, ORIGINE[path] = path, "dal disco"
    copia = _copia_di(path)
    if copia:
        if _copia_buona(path, copia):
            sorgente, ORIGINE[path] = copia, "dalla copia locale"
        elif _spazio_per(path):
            _copia_dal_bucket(path, copia, ferma)
            sorgente, ORIGINE[path] = copia, "scaricata nella copia locale"
        else:
            ORIGINE[path] = "dal bucket, senza copia"
    fd, tmp = tempfile.mkstemp(prefix="carica_", suffix=".h5")
    os.close(fd)
    try:
        _travasa(sorgente, tmp, ferma, apri_da=gzip.open)
    except InterruptedError:
        os.unlink(tmp)
        raise
    except BaseException as e:
        os.unlink(tmp)
        if sorgente == copia and isinstance(e, (OSError, EOFError, zlib.error)):
            # Una copia che non si decomprime e' rovinata: via, la prossima
            # volta si riscarica dal bucket
            os.unlink(copia)
            raise OSError("la copia locale di %s era rovinata (%s): cancellata, rieseguendo si "
                          "riscarica" % (os.path.basename(path), e)) from e
        raise
    return tmp, tmp


@contextlib.contextmanager
def apri(path):
    """(file h5py, header) di un file gia' in locale. Una run in corso, o
    chiusa male, si apre in SWMR."""
    try:
        f = h5py.File(path, "r")
    except OSError:
        f = h5py.File(path, "r", libver="latest", swmr=True)
    try:
        if "/events/waveforms" not in f:
            raise ValueError("formato v1 (un dataset per evento), non letto qui")
        yield f, daqio.read_header(f)
    finally:
        f.close()


def geometria(hdr):
    """(campioni per canale, passo in ns, {canale: colonne della riga})."""
    ns = int(hdr["SamplesPerChannel"])
    if "SamplingTime" in hdr:
        dt = float(hdr["SamplingTime"]) * 1e9
    else:  # file vecchi: '750MHz', '1GHz'
        m = re.match(r"([0-9.]+)\s*([GM])", str(hdr["SamplingRate"]))
        dt = 1000.0 / (float(m.group(1)) * (1000 if m.group(2) == "G" else 1))
    nel_file = [int(c) for c in hdr["ChannelList"]]
    mancanti = [c for c in CANALI if c not in nel_file]
    if mancanti:
        raise ValueError("Canali non presenti nel file: %s (ci sono %s)" % (mancanti, nel_file))
    return ns, dt, {c: slice(nel_file.index(c) * ns, (nel_file.index(c) + 1) * ns) for c in CANALI}


def stessa_presa(hdr, rif, nome):
    """Con frequenza, campioni o PostTrigger diversi l'impulso cade altrove, e
    i cancelli della prima run non valgono: la run va analizzata a parte."""
    for k in ("SamplingRate", "SamplesPerChannel", "PostTriggerSize"):
        if k in rif and str(hdr.get(k)) != str(rif[k]):
            raise ValueError("%s: %s = %s, nella prima run %s. Toglierla dalla lista, o "
                             "analizzarla a parte." % (nome, k, hdr.get(k), rif[k]))


def verso_canale(c, info, y):
    """+1 / -1, e chi l'ha deciso: a mano, il log, o -- senza log -- la
    posizione del piedistallo, che e' cio' che il DCOffset decide: in basso
    per impulsi positivi, in alto per negativi."""
    if c in VERSO:
        return VERSO[c], "a mano"
    v = info["versi"] if info else None
    if isinstance(v, dict) and v.get(c) in (1, -1):
        return v[c], "dal DCOffset nel log"
    if v in (1, -1):
        return v, "da TriggerPolarity nel log"
    base = float(np.median(y))
    return (1 if base < 2048 else -1), "dal piedistallo a %.0f conteggi: log assente" % base


def controlla_versi(info, nome):
    """Il verso di un canale non deve cambiare fra le run di una lista."""
    if info is None or info["versi"] is None:
        return
    for c in CANALI:
        v = info["versi"].get(c) if isinstance(info["versi"], dict) else info["versi"]
        if c not in VERSO and v in (1, -1) and v != VERSI[c]:
            raise ValueError("%s: ch%d ha impulsi %s, nella prima run %s. Le run di una lista "
                             "devono avere gli stessi DCOffset." % (
                                 nome, c, "positivi" if v > 0 else "negativi",
                                 "positivi" if VERSI[c] > 0 else "negativi"))


def chiusa(path, info):
    """Una run finita: un .gz (archivia.py comprime solo quelle) o col riepilogo nel log."""
    return path.endswith(".gz") or (info is not None and info["durata_s"] is not None)


def leggi_campione(path, info, n):
    """(header, primi n eventi dei canali analizzati, passo in ns, dalla cache?).
    In cache, perche' dal bucket montato la prima run costa minuti; la copia
    scaricata qui resta in LOCALI per la lettura vera, che non la riscarica."""
    cache = os.path.join(CACHE, "%s__campione_%d.npz" % (efficienza.nome_run(path), n))
    if os.path.exists(cache):
        with np.load(cache) as z:
            if [int(c) for c in z["canali"]] == list(CANALI):
                return json.loads(str(z["hdr"])), z["y"], float(z["dt"]), True
    if path not in LOCALI:
        LOCALI[path] = scarica(path)
    try:
        with apri(LOCALI[path][0]) as (f, hdr):
            _, dt, col = geometria(hdr)
            ds = f["/events/waveforms"]
            w = np.asarray(ds[:min(n, ds.shape[0])])
    except BaseException:
        _, tmp = LOCALI.pop(path)
        if tmp:
            os.unlink(tmp)
        raise
    y = np.stack([w[:, col[c]] for c in CANALI], axis=1)
    if chiusa(path, info):
        os.makedirs(CACHE, exist_ok=True)
        semplice = {k: (v.tolist() if isinstance(v, np.ndarray) else
                        v.item() if isinstance(v, np.generic) else v) for k, v in hdr.items()}
        np.savez_compressed(cache + ".tmp.npz", y=y, dt=dt, canali=np.array(CANALI),
                            hdr=json.dumps(semplice))
        os.replace(cache + ".tmp.npz", cache)
    return hdr, y, dt, False


def regola(c):
    return CANCELLO_AUTO[RIVELATORE[c]]


def metodo(c):
    return PIEDISTALLO[RIVELATORE[c]]["metodo"]


def soglia_forma(c):
    return SOGLIA_FORMA[RIVELATORE[c]]


def finestra_auto(y, segno, t_ns, r, soglia_forma):
    """(inizio, fine) del cancello in ns ed eventi usati, dalla forma d'onda
    media degli eventi col massimo sopra soglia_forma: dove scende sotto
    r["soglia"] del picco, meno r["prima_ns"] all'inizio; alla fine piu'
    r["dopo_ns"], o in fondo alla traccia meno r["fine_traccia"] campioni.
    Con soglia 0.10, 10 e 20 ns e' la regola del monitor e di
    carica_padme.finestre, che pero' mediava il 20% di eventi piu' alti: sui
    PMT, con un impulso vero in ~7 eventi su 100, 6 su 10 erano solo rumore.
    La base e' la mediana della traccia: per scegliere gli eventi basta.
    None se gli eventi sopra soglia sono meno di 20, o la forma media non
    supera 5 volte il rumore."""
    x = segno * (y - np.median(y, axis=1, keepdims=True))
    amp = x[:, :x.shape[1] - CAMPIONI_FINALI_ESCLUSI].max(axis=1)
    sopra = amp >= soglia_forma
    if sopra.sum() < 20:
        return None
    m = x[sopra].mean(axis=0)
    p = int(np.argmax(m))
    picco = float(m[p])
    rumore = float(np.median(np.std(x[:, :max(MIN_CAMPIONI_PED, p // 2)], axis=1)))
    if picco < 5 * max(rumore, 0.5):
        return None
    a = b = p
    while a > 0 and m[a] > r["soglia"] * picco:
        a -= 1
    while b < len(m) - 1 and m[b] > r["soglia"] * picco:
        b += 1
    inizio = max(0.0, float(t_ns[a]) - r["prima_ns"])
    if r.get("fine_traccia"):
        fine = float(t_ns[len(t_ns) - r["fine_traccia"]])
    else:
        fine = min(float(t_ns[-1]), float(t_ns[b]) + r["dopo_ns"])
    return inizio, fine, int(sopra.sum())


def finestre_canale(c, y, segno, t_ns):
    """Cancello (ns) e piedistallo (indici) di un canale, e come e' stato deciso il cancello."""
    r = regola(c)
    n_forma = None
    if c in CANCELLO:
        canc, come, come_it = tuple(float(x) for x in CANCELLO[c]), "by hand", "a mano"
    else:
        auto = finestra_auto(y.astype(np.float32), segno, t_ns, r, soglia_forma(c))
        if auto is None:
            raise RuntimeError(
                "ch%d: nessun impulso riconoscibile nei primi %d eventi (meno di 20 sopra "
                "SOGLIA_FORMA = %g conteggi, o forma media sotto 5 volte il rumore). Abbassare la "
                "soglia, dare il cancello a mano, CANCELLO[%d] = (da_ns, a_ns), o togliere il "
                "canale da CANALI." % (c, len(y), soglia_forma(c), c))
        canc, n_forma = auto[:2], auto[2]
        if r.get("fine_traccia"):
            fine_en, fine_it = ("trace end -%d samples" % r["fine_traccia"],
                                "fino in fondo meno %d campioni" % r["fine_traccia"])
        else:
            fine_en = fine_it = "+%g ns" % r["dopo_ns"]
        come = "auto: %g%% of peak, -%g ns / %s" % (100 * r["soglia"], r["prima_ns"], fine_en)
        come_it = "automatico: %g%% del picco, -%g ns / %s" % (100 * r["soglia"], r["prima_ns"], fine_it)
    if c in FINESTRA_PED:
        pi = tuple(int(np.searchsorted(t_ns, x)) for x in FINESTRA_PED[c])
    else:
        pd = PIEDISTALLO[RIVELATORE[c]]
        pi = (int(np.searchsorted(t_ns, pd["inizio_ns"])),
              int(np.searchsorted(t_ns, canc[0] - pd["distacco_ns"])))
    if pi[1] - pi[0] < MIN_CAMPIONI_PED:
        raise RuntimeError("ch%d: cancello a %.0f ns, troppo presto per un piedistallo prima: "
                           "darlo a mano in FINESTRA_PED[%d]" % (c, canc[0], c))
    gi = tuple(int(np.searchsorted(t_ns, x)) for x in canc)
    return {"cancello_ns": canc, "gi": gi, "pi": pi, "come": come, "come_it": come_it,
            "n_forma": n_forma}


def chiave_cache(fin):
    return json.dumps({"versione": VERSIONE_CACHE, "max": MAX_EVENTI_PER_RUN,
                       "finestre": {str(c): [fin[c]["gi"], fin[c]["pi"]] for c in CANALI}},
                      sort_keys=True)


def dalla_cache(path, info, fin):
    """Le somme di una run chiusa gia' letta con queste finestre, o None:
    cambiato un cancello, la run si rilegge."""
    cache = os.path.join(CACHE, efficienza.nome_run(path) + ".npz")
    if not (chiusa(path, info) and os.path.exists(cache)):
        return None
    with np.load(cache) as z:
        if str(z["chiave"]) != chiave_cache(fin):
            return None
        r = {k: z[k] for k in z.files if k != "chiave"}
    r["n"], r["n_file"] = int(r["n"]), int(r["n_file"])
    return r


def metti_in_cache(path, fin, r):
    os.makedirs(CACHE, exist_ok=True)
    cache = os.path.join(CACHE, efficienza.nome_run(path) + ".npz")
    np.savez_compressed(cache + ".tmp.npz", chiave=chiave_cache(fin), **r)
    os.replace(cache + ".tmp.npz", cache)


def calcola(locale, nome, fin, rif):
    """Per ogni canale, evento per evento: somma dei campioni nel cancello,
    media della finestra del piedistallo, saturazione. A blocchi di ~50 MB:
    una run da 100000 eventi tutta insieme sarebbe 1 GB."""
    with apri(locale) as (f, hdr):
        stessa_presa(hdr, rif, nome)
        _, _, col = geometria(hdr)
        ds = f["/events/waveforms"]
        # A run in corso i dataset possono differire di qualche riga
        n_file = min([ds.shape[0]] + [f[k].shape[0] for k in
                                      ("/events/event_counter", "/events/trigger_time_tag") if k in f])
        n = min(n_file, MAX_EVENTI_PER_RUN or n_file)
        r = {"n": n, "n_file": n_file}
        for c in CANALI:
            r["S_ch%d" % c] = np.empty(n)
            r["P_ch%d" % c] = np.empty(n, np.float32)
            r["sat_ch%d" % c] = np.empty(n, bool)
            for k in ("max", "min", "imax", "imin"):
                r["%s_ch%d" % (k, c)] = np.empty(n, np.int16)
        passo = max(256, int(5e7 // (2 * ds.shape[1])))
        for a in range(0, n, passo):
            b = min(n, a + passo)
            w = np.asarray(ds[a:b])
            for c in CANALI:
                y = w[:, col[c]]
                (g0, g1), (p0, p1) = fin[c]["gi"], fin[c]["pi"]
                yg = y[:, g0:g1]
                r["S_ch%d" % c][a:b] = yg.sum(axis=1, dtype=np.float64)
                r["P_ch%d" % c][a:b] = y[:, p0:p1].mean(axis=1, dtype=np.float64)
                r["sat_ch%d" % c][a:b] = (yg.max(axis=1) >= FONDO_SCALA) | (yg.min(axis=1) <= 0)
                r["max_ch%d" % c][a:b] = yg.max(axis=1)
                r["min_ch%d" % c][a:b] = yg.min(axis=1)
                r["imax_ch%d" % c][a:b] = yg.argmax(axis=1)
                r["imin_ch%d" % c][a:b] = yg.argmin(axis=1)
        for k, d in (("event_counter", "/events/event_counter"), ("ttt", "/events/trigger_time_tag")):
            r[k] = np.asarray(f[d][:n]) if d in f else np.zeros(0, np.uint32)
    return r


def durata(r, info):
    """Tempo di acquisizione in s, e da dove viene: quello che la DAQ dichiara
    nel log, se si e' letta la run intera; se no l'arco del Trigger Time Tag."""
    if info and info["durata_s"] is not None and r["n"] == r["n_file"]:
        return info["durata_s"], "log"
    if len(r["ttt"]) > 1:
        passi = np.diff(r["ttt"].astype(np.int64)) % TTT_GIRO
        return float(passi.sum()) * TTT_NS * 1e-9, "TTT"
    return float("nan"), "?"


def grandezze(r, c, segno, fin, dt):
    """Per i due piedistalli ("evento", "blocco"): carica in ADC*ns e ampiezza
    nel cancello in mV, positive nel verso del canale; e l'istante del massimo
    in ns. {"q_evento", "q_blocco", "a_evento", "a_blocco", "t"}."""
    S = r["S_ch%d" % c]
    P = r["P_ch%d" % c].astype(np.float64)
    g0, g1 = fin[c]["gi"]
    # Un resto piu' corto di mezzo blocco va col blocco prima: la mediana di
    # pochi eventi ballerebbe.
    bordi = list(range(0, len(P), EVENTI_PER_BLOCCO)) + [len(P)]
    if len(bordi) > 2 and bordi[-1] - bordi[-2] < EVENTI_PER_BLOCCO // 2:
        del bordi[-2]
    P_blocco = np.concatenate([np.full(b - a, np.median(P[a:b]))
                               for a, b in zip(bordi[:-1], bordi[1:])])
    estremo = (r["max_ch%d" % c] if segno > 0 else r["min_ch%d" % c]).astype(np.float64)
    i_estremo = r["imax_ch%d" % c] if segno > 0 else r["imin_ch%d" % c]
    fuori = {"t": ((g0 + i_estremo.astype(np.float64)) * dt).astype(np.float32)}
    for m, ped in (("evento", P), ("blocco", P_blocco)):
        fuori["q_" + m] = (segno * (S - ped * (g1 - g0)) * dt).astype(np.float32)
        fuori["a_" + m] = (segno * (estremo - ped) * MV_PER_CONTEGGIO).astype(np.float32)
    return fuori


# --- la selezione: espressioni lette con l'albero sintattico di Python e
# valutate su colonne di numpy, con solo le operazioni ammesse (niente eval)
import ast
_CONFRONTI = {ast.Gt: np.greater, ast.Lt: np.less, ast.GtE: np.greater_equal,
              ast.LtE: np.less_equal, ast.Eq: np.equal, ast.NotEq: np.not_equal}
_OPERAZIONI = {ast.Add: np.add, ast.Sub: np.subtract, ast.Mult: np.multiply, ast.Div: np.divide}


def _conta(condizioni):
    return np.sum([np.asarray(x, bool) for x in condizioni], axis=0)


FUNZIONI = {
    "almeno": lambda n, *c: _conta(c) >= n,
    "al_piu": lambda n, *c: _conta(c) <= n,
    "esattamente": lambda n, *c: _conta(c) == n,
    "abs": np.abs,
    "min": lambda *x: np.minimum.reduce([np.asarray(v, float) for v in x]),
    "max": lambda *x: np.maximum.reduce([np.asarray(v, float) for v in x]),
}


def valuta(espr, var):
    """Il valore dell'espressione `espr` sulle colonne di `var`, evento per evento."""
    def v(n):
        if isinstance(n, ast.Expression):
            return v(n.body)
        if isinstance(n, ast.BoolOp):
            vals = [np.asarray(v(x), bool) for x in n.values]
            return (np.logical_and if isinstance(n.op, ast.And) else np.logical_or).reduce(vals)
        if isinstance(n, ast.UnaryOp) and isinstance(n.op, ast.Not):
            return ~np.asarray(v(n.operand), bool)
        if isinstance(n, ast.UnaryOp) and isinstance(n.op, (ast.USub, ast.UAdd)):
            return -v(n.operand) if isinstance(n.op, ast.USub) else v(n.operand)
        if isinstance(n, ast.Compare) and all(type(o) in _CONFRONTI for o in n.ops):
            sx, esito = v(n.left), True
            for op, nodo in zip(n.ops, n.comparators):
                dx = v(nodo)
                esito = esito & _CONFRONTI[type(op)](sx, dx)
                sx = dx
            return esito
        if isinstance(n, ast.BinOp) and type(n.op) in _OPERAZIONI:
            return _OPERAZIONI[type(n.op)](v(n.left), v(n.right))
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float, bool)):
            return n.value
        if isinstance(n, ast.Name):
            if n.id in var:
                return var[n.id]
            m = re.fullmatch(r"Q(\d+)", n.id)
            if m and int(m.group(1)) in CANALI:
                raise NameError("%s: per la carica in PE di ch%s serve PE_ADCNS[%s] (ADC*ns per "
                                "fotoelettrone); in ADC*ns e' C%s" % ((n.id,) + (m.group(1),) * 3))
            raise NameError("%r sconosciuto. Variabili: %s; condizioni: %s" % (
                n.id, ", ".join(k for k in var if k[0] in "ACQST" and k[1:].isdigit()),
                ", ".join(k for k in var if not (k[0] in "ACQST" and k[1:].isdigit())) or "nessuna"))
        if isinstance(n, ast.Call) and isinstance(n.func, ast.Name) and n.func.id in FUNZIONI \
                and not n.keywords:
            return FUNZIONI[n.func.id](*[v(x) for x in n.args])
        raise ValueError("non ammesso nella selezione: %r" % ast.unparse(n))
    try:
        albero = ast.parse(espr.strip(), mode="eval")
    except SyntaxError as e:
        raise SyntaxError("selezione scritta male: %r (%s)" % (espr, e.msg)) from None
    return v(albero)


def condizione(x, nome, n):
    """La colonna vero/falso di una condizione, o un errore che dice perche' non lo e'."""
    x = np.asarray(x)
    if x.ndim == 0:
        x = np.full(n, x)
    if x.dtype != bool:
        raise ValueError("%s da' un numero, non vero/falso: manca un confronto (>, <, ...)?" % nome)
    return x


def picco_a_zero(q, giri=30):
    """Centro e larghezza del picco degli eventi senza impulso: mediana e
    mezza distanza fra 16 e 84 percentile degli eventi entro +-3 sigma,
    ripetendo finche' si assesta. Non una deviazione standard su tutto, che
    sui SiPM prende la gobba degli impulsi veri, e non la FWHM di un
    istogramma fine: con pochi eventi segue le fluttuazioni da un intervallo
    all'altro, e le cariche vanno a passi di un campione (1.333 ADC*ns), che
    gli intervalli contengono ora 3 ora 4 volte (su 5121 eventi dava 12 ADC*ns
    invece di ~170). Sui SiPM conta anche le spalle del picco (impulsi al buio,
    errori del piedistallo), quindi viene piu' larga della FWHM del nucleo."""
    x = np.asarray(q, np.float64)
    h, e = np.histogram(x, bins=100, range=tuple(np.percentile(x, [0.5, 99.5])))
    k = int(np.argmax(h))
    m, s = 0.5 * (e[k] + e[k + 1]), float(e[1] - e[0])
    for _ in range(giri):
        dentro = x[np.abs(x - m) < 3 * s]
        if len(dentro) < 20:
            break
        p16, m2, p84 = np.percentile(dentro, [16, 50, 84])
        s2 = 0.5 * (p84 - p16)
        fermo = abs(m2 - m) < 1e-3 * s2 and abs(s2 - s) < 1e-3 * s2
        m, s = float(m2), float(s2)
        if fermo:
            break
    return m, s

In [ ]:
## === Le run scelte ===
FILE = trova_file(RUN)
NOMI_FILE = [efficienza.nome_run(p) for p in FILE]
_p, _lab, _s = etichette(NOMI_FILE)
NOME_USCITA = (NOMI_FILE[0] if len(FILE) == 1
               else "%s%s-%s%s_%drun" % (_p, _lab[0], _lab[-1], _s, len(FILE)))

INFO = {p: leggi_log(p) for p in FILE}

print("%d run in %s (dai log):" % (len(FILE), os.path.dirname(FILE[0])))
for p in FILE:
    info = INFO[p]
    if p.endswith(".h5") and os.path.realpath(p).startswith(REMOTO + os.sep):
        stato = "sul cloud solo il .h5, copia presa a run in corso o chiusa male: si salta"
    elif info is None:
        stato = "senza log: durata dal Trigger Time Tag, verso dal piedistallo"
    elif info["durata_s"] is None:
        stato = "in corso, o chiusa male: si legge quello che c'e'"
    else:
        stato = "%6s eventi in %5.0f s%s" % (info["eventi"] if info["eventi"] is not None else "?",
                                              info["durata_s"],
                                              ", collegamento perso" if info["collegamento_perso"] else "")
    print("  %-38s %s" % (os.path.basename(p), stato))

In [ ]:
## === Cancello e piedistallo, dalla prima run della lista ===
HDR_RIF, Y_RIF, DT, _dalla_cache = leggi_campione(FILE[0], INFO[FILE[0]], EVENTI_PER_FINESTRE)
T_NS = np.arange(Y_RIF.shape[2]) * DT
INFO_RIF = INFO[FILE[0]]
VERSI, FIN = {}, {}
print("Dai primi %d eventi di %s%s, a %.3f ns per campione:"
      % (len(Y_RIF), NOMI_FILE[0], " (dalla cache)" if _dalla_cache
         else " (%s)" % ORIGINE.get(FILE[0], "?"), DT))
for j, c in enumerate(CANALI):
    VERSI[c], come = verso_canale(c, INFO_RIF, Y_RIF[:, j, :])
    FIN[c] = finestre_canale(c, Y_RIF[:, j, :], VERSI[c], T_NS)
    g, p = FIN[c]["cancello_ns"], FIN[c]["pi"]
    print("  ch%d %-4s impulsi %s (%s); cancello %.1f-%.1f ns (%s); piedistallo %.1f-%.1f ns, %s"
          % (c, RIVELATORE.get(c, ""), "positivi" if VERSI[c] > 0 else "negativi", come,
             g[0], g[1], FIN[c]["come_it"] + (", forma media di %d eventi sopra %g conteggi"
                                               % (FIN[c]["n_forma"], soglia_forma(c)) if FIN[c]["n_forma"] else ""),
             T_NS[p[0]], T_NS[p[1] - 1], "per " + metodo(c)))
print("\nPer tenerli fissi qualunque sia la prima run, nella CONFIGURAZIONE:")
print("CANCELLO = {%s}" % ", ".join("%d: (%.1f, %.1f)" % (c, *FIN[c]["cancello_ns"]) for c in CANALI))

In [ ]:
## === Grafico: su cosa si sono decisi cancello e piedistallo ===
# Le forme d'onda con la loro polarita' vera; a destra dove cade il massimo
# degli stessi eventi: un impulso fuori dal cancello si vede li'.
os.makedirs(USCITA, exist_ok=True)
rng = np.random.default_rng(1)
fig, assi = plt.subplots(len(CANALI), 2, figsize=(14, 2.8 * len(CANALI)), squeeze=False,
                         gridspec_kw={"width_ratios": [1.7, 1]})
for (a, b), (j, c) in zip(assi, enumerate(CANALI)):
    g, p = FIN[c]["cancello_ns"], FIN[c]["pi"]
    # La base e' la finestra del piedistallo, non la mediana della traccia: un
    # impulso di SiPM occupa meta' traccia e la mediana la alza di ~30 conteggi.
    y = Y_RIF[:, j, :].astype(np.float32)
    y -= y[:, p[0]:p[1]].mean(axis=1, keepdims=True)
    fino = y.shape[1] - CAMPIONI_FINALI_ESCLUSI
    amp = (VERSI[c] * y[:, :fino]).max(axis=1)
    alti = np.flatnonzero(amp >= soglia_forma(c))
    for k in rng.choice(alti, size=min(30, len(alti)), replace=False):
        a.plot(T_NS, y[k], lw=0.4, color="#c3cbd8")
    a.plot(T_NS, y[alti].mean(axis=0), lw=1.4, color="#1f4e9c",
           label="mean of the %d events above %g counts, 30 of them in grey" % (len(alti), soglia_forma(c)))
    a.axvspan(*g, color="#e8710a", alpha=0.18, label="gate %.0f-%.0f ns (%s)" % (g[0], g[1], FIN[c]["come"]))
    a.axvspan(T_NS[p[0]], T_NS[p[1] - 1], color="#888888", alpha=0.18,
              label="pedestal %.0f-%.0f ns" % (T_NS[p[0]], T_NS[p[1] - 1]))
    a.set_title("ch%d %s - first %d events of %s" % (c, RIVELATORE.get(c, ""), len(y), NOMI_FILE[0]),
                fontsize=9)
    a.set_xlabel("time [ns]")
    a.set_ylabel("counts above pedestal")
    a.legend(fontsize=7, loc="upper right")
    a.grid(alpha=0.25)

    b.hist(T_NS[np.argmax(VERSI[c] * y[alti, :fino], axis=1)],
           bins=np.arange(0, T_NS[-1] + 8 * DT, 8 * DT), color="#1f4e9c")
    b.axvspan(*g, color="#e8710a", alpha=0.18)
    b.set_title("ch%d - time of the maximum, same events" % c, fontsize=9)
    b.set_xlabel("time [ns]")
    b.set_ylabel("events")
    b.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(os.path.join(USCITA, "carica_run_%s_finestre.png" % NOME_USCITA), dpi=110)
plt.show()

In [ ]:
## === Lettura delle run ===
# Le run non ancora in cache si scaricano SCARICHI_IN_PARALLELO alla volta, in
# anticipo, mentre si leggono quelle arrivate: dal bucket montato un file solo
# va a 1.4-4 MB/s, tre insieme a ~7. Una run che non si legge si salta e si
# dice: su una lista lunga non deve buttare via quello che si e' gia' scaricato.
FERMA = threading.Event()
DALLA_CACHE = {p: dalla_cache(p, INFO[p], FIN) for p in FILE}
da_scaricare = [p for p in FILE if DALLA_CACHE[p] is None and p not in LOCALI]
in_volo, prossimo = {}, 0
pool = concurrent.futures.ThreadPoolExecutor(max_workers=max(1, SCARICHI_IN_PARALLELO))


def riempi():
    global prossimo
    while len(in_volo) < max(1, SCARICHI_IN_PARALLELO) and prossimo < len(da_scaricare):
        in_volo[da_scaricare[prossimo]] = pool.submit(scarica, da_scaricare[prossimo], FERMA)
        prossimo += 1


print("%d run: %d dalla cache, %d da leggere" % (
    len(FILE), sum(r is not None for r in DALLA_CACHE.values()),
    sum(r is None for r in DALLA_CACHE.values())))
RISULTATI, SALTATE = [], []
inizio = time.time()
try:
    for k, path in enumerate(FILE):
        riempi()
        nome = efficienza.nome_run(path)
        info = INFO[path]
        t1 = time.time()
        try:
            controlla_versi(info, nome)
            r, come = DALLA_CACHE[path], "cache"
            if r is None:
                locale, tmp = LOCALI.pop(path) if path in LOCALI else in_volo.pop(path).result()
                attesa = time.time() - t1
                try:
                    r = calcola(locale, nome, FIN, HDR_RIF)
                finally:
                    if tmp:
                        os.unlink(tmp)
                if chiusa(path, info):
                    metti_in_cache(path, FIN, r)
                come = "%s, attesa %.0f s, letta in %.0f s" % (ORIGINE.get(path, "?"), attesa,
                                                              time.time() - t1 - attesa)
            if r["n"] == 0:
                raise ValueError("nessun evento")
        except Exception as e:
            SALTATE.append({"run": nome, "motivo": "%s: %s" % (type(e).__name__, e)})
            print("[%2d/%d] %-34s SALTATA: %s" % (k + 1, len(FILE), nome, SALTATE[-1]["motivo"]))
            continue
        d, fonte = durata(r, info)
        note = []
        if info is None:
            note.append("senza log")
        else:
            if info["durata_s"] is None:
                note.append("in corso o chiusa male")
            if info["collegamento_perso"]:
                note.append("collegamento perso")
        if r["n"] < r["n_file"]:
            note.append("letti %d di %d" % (r["n"], r["n_file"]))
        RISULTATI.append({"nome": nome, "path": path, "r": r, "durata_s": d, "fonte": fonte,
                          "drs4": info["drs4"] if info else "?", "note": note})
        print("[%2d/%d] %-34s %7d eventi in %6.0f s (%s), %5.1f Hz   %s%s"
              % (k + 1, len(FILE), nome, r["n"], d, fonte, r["n"] / d if d else float("nan"),
                 come, "  (" + ", ".join(note) + ")" if note else ""))
finally:
    # Anche se interrotta: niente scaricamenti appesi ne' file da 1 GB lasciati indietro
    FERMA.set()
    pool.shutdown(wait=True, cancel_futures=True)
    avanzi = [f.result()[1] for f in in_volo.values()
              if f.done() and not f.cancelled() and f.exception() is None]
    avanzi += [tmp for _, tmp in LOCALI.values()]
    for tmp in avanzi:
        if tmp and os.path.exists(tmp):
            os.unlink(tmp)
    LOCALI.clear()
print("lette in %.0f s%s" % (time.time() - inizio, "; SALTATE %d: %s" % (
    len(SALTATE), ", ".join(x["run"] for x in SALTATE)) if SALTATE else ""))
if not RISULTATI:
    raise RuntimeError("nessuna run letta")

NOMI = [x["nome"] for x in RISULTATI]
DURATE = np.array([x["durata_s"] for x in RISULTATI])
RUN_IDX = np.concatenate([np.full(x["r"]["n"], i, np.int32) for i, x in enumerate(RISULTATI)])
EVENTO = np.concatenate([np.arange(x["r"]["n"], dtype=np.int32) for x in RISULTATI])
Q, AMP, TMAX, SAT, ZERO = {}, {}, {}, {}, {}
for c in CANALI:
    parti = [grandezze(x["r"], c, VERSI[c], FIN, DT) for x in RISULTATI]
    Q[c] = {m: np.concatenate([g["q_" + m] for g in parti]) for m in ("evento", "blocco")}
    AMP[c] = {m: np.concatenate([g["a_" + m] for g in parti]) for m in ("evento", "blocco")}
    TMAX[c] = np.concatenate([g["t"] for g in parti])
    SAT[c] = np.concatenate([x["r"]["sat_ch%d" % c] for x in RISULTATI])
    ZERO[c] = {m: picco_a_zero(Q[c][m]) for m in Q[c]}

print("\n%-6s %-5s %-14s %-14s %-7s %8s %7s   picco a zero, mediana +- (p84-p16)/2 [ADC*ns]: per evento / per blocco"
      % ("canale", "", "cancello [ns]", "piedist. [ns]", "scelto", "eventi", "saturi"))
for c in CANALI:
    g, p = FIN[c]["cancello_ns"], FIN[c]["pi"]
    print("ch%-4d %-5s %5.0f - %-6.0f %5.0f - %-6.0f %-7s %8d %7d   %+.0f +- %.0f / %+.0f +- %.0f"
          % (c, RIVELATORE.get(c, ""), g[0], g[1], T_NS[p[0]], T_NS[p[1] - 1],
             metodo(c), len(SAT[c]), SAT[c].sum(),
             *ZERO[c]["evento"], *ZERO[c]["blocco"]))

In [ ]:
## === Selezione degli eventi ===
# Per cambiare CONDIZIONI o SELEZIONE: rieseguire la CONFIGURAZIONE e poi da qui
# in giu'. Le run non si rileggono.
risolvi_cartelle()
N_EV = len(RUN_IDX)
VAR = {}
for c in CANALI:
    m = metodo(c)
    VAR["C%d" % c], VAR["A%d" % c] = Q[c][m], AMP[c][m]
    VAR["T%d" % c], VAR["S%d" % c] = TMAX[c], SAT[c]
    if c in PE_ADCNS:
        VAR["Q%d" % c] = Q[c][m] / PE_ADCNS[c]
ESITI = {}
for nome, espr in CONDIZIONI.items():
    ESITI[nome] = condizione(valuta(espr, VAR), nome, N_EV)
    VAR[nome] = ESITI[nome]
ATTIVA = bool(SELEZIONE.strip())
SEL = condizione(valuta(SELEZIONE, VAR), "SELEZIONE", N_EV) if ATTIVA else np.ones(N_EV, bool)
NOME_SEL = NOME_USCITA + ("_sel-" + re.sub(r"[^A-Za-z0-9]+", "-", SELEZIONE).strip("-")[:60]
                          if ATTIVA else "")

T_TOT = DURATE.sum()
print("%d eventi in %.0f s" % (N_EV, T_TOT))
for nome, esito in ESITI.items():
    print("  %-16s %8d eventi (%5.1f%%, %7.3f Hz)   %s"
          % (nome, esito.sum(), 100 * esito.mean(), esito.sum() / T_TOT, CONDIZIONI[nome]))
print("SELEZIONE: %s" % ("%r -> %d eventi (%.2f%%, %.3f Hz)"
                         % (SELEZIONE, SEL.sum(), 100 * SEL.mean(), SEL.sum() / T_TOT)
                         if ATTIVA else "nessuna, tutti gli eventi"))

In [ ]:
## === Funzioni dei grafici ===
ALTRO = {"evento": "blocco", "blocco": "evento"}
DESCR = {"evento": "pedestal per event",
         "blocco": "pedestal per block of %d events" % EVENTI_PER_BLOCCO}


def intervallo(c, q):
    """L'asse delle cariche: XLIM se c'e', se no dallo 0.05 al 99.95 percentile
    (il titolo del pannello dice quanti eventi restano fuori)."""
    if c in XLIM:
        return XLIM[c]
    lo, hi = np.percentile(q, [0.05, 99.95])
    return lo - 0.05 * (hi - lo), hi + 0.05 * (hi - lo)


def pannelli(titolo):
    """Una griglia a due colonne riempita per colonne: con i cinque canali del
    LNGS i SiPM stanno a sinistra, i PMT a destra, e il sesto riquadro resta
    libero per la legenda o il riepilogo."""
    nr = (len(CANALI) + 1) // 2
    fig, assi = plt.subplots(nr, 2, figsize=(14, 3.6 * nr), squeeze=False)
    fig.suptitle(titolo, fontsize=11)
    return fig, [assi[i % nr, i // nr] for i in range(len(CANALI))], \
        (assi[-1, -1] if len(CANALI) % 2 else None)

In [ ]:
## === Distribuzioni di carica ===
# Con una selezione attiva la curva grigia sono tutti gli eventi; senza, la
# carica con l'altro piedistallo. L'asse e' sempre quello di tutti gli eventi,
# cosi' selezioni diverse si confrontano a occhio.
fig, assi, libero = pannelli("%s - charge per channel%s" % (
    NOMI[0] if len(NOMI) == 1 else "%d runs, %s ... %s" % (len(NOMI), NOMI[0], NOMI[-1]),
    " - selection: " + SELEZIONE if ATTIVA else ""))
for a, c in zip(assi, CANALI):
    scelto = metodo(c)
    tutti = Q[c][scelto]
    q, sat = tutti[SEL], SAT[c][SEL]
    lo, hi = intervallo(c, tutti)
    bins = np.linspace(lo, hi, N_BIN + 1)
    fuori = int(((q < lo) | (q > hi)).sum())
    if ATTIVA:
        a.hist(tutti, bins, histtype="step", color="#9a9a9a", lw=0.9,
               label="all events (%d), %s" % (len(tutti), DESCR[scelto]))
        a.hist(q, bins, histtype="step", color="#1f4e9c", lw=1.4,
               label="selected (%d, %.2f%%)" % (len(q), 100.0 * len(q) / len(tutti)))
    else:
        a.hist(Q[c][ALTRO[scelto]], bins, histtype="step", color="#9a9a9a", lw=0.9,
               label="%s  (zero peak: σ = %.0f)" % (DESCR[ALTRO[scelto]], ZERO[c][ALTRO[scelto]][1]))
        a.hist(q, bins, histtype="step", color="#1f4e9c", lw=1.4,
               label="%s  (zero peak: σ = %.0f)" % (DESCR[scelto], ZERO[c][scelto][1]))
    if sat.any():
        a.hist(q[sat], bins, color="#e8710a", alpha=0.5,
               label="saturated: %d events (charge is a lower limit)" % sat.sum())
    g = FIN[c]["cancello_ns"]
    a.set_title("ch%d %s - gate %.0f-%.0f ns, %d events%s" % (
        c, RIVELATORE.get(c, ""), g[0], g[1], len(q), ", %d outside the axis" % fuori if fuori else ""),
        fontsize=9.5)
    a.set_yscale("log")
    a.set_xlim(lo, hi)
    a.set_xlabel("charge [ADC·ns]   (1 pC = 204.8 ADC·ns)")
    a.set_ylabel("events / %.0f ADC·ns" % (bins[1] - bins[0]))
    a.legend(fontsize=7.5, frameon=False)
    a.grid(alpha=0.25)
if libero is not None:
    libero.axis("off")
    drs4 = sorted({x["drs4"] for x in RISULTATI})
    righe = ["%d run%s, %d events, %.2f h of acquisition (%.1f Hz)" % (
                 len(NOMI), "s" if len(NOMI) > 1 else "", len(RUN_IDX), DURATE.sum() / 3600,
                 len(RUN_IDX) / DURATE.sum()),
             "first: %s" % NOMI[0]] + (["last:  %s" % NOMI[-1]] if len(NOMI) > 1 else []) + [
             "gates and pedestals from the first %d events of the first run" % len(Y_RIF),
             "sampling %.3f ns; CAEN DRS4 correction %s; no PADME" % (DT, "/".join(drs4))] + (
             ["SKIPPED: " + ", ".join(x["run"] for x in SALTATE)] if SALTATE else [])
    if ATTIVA:
        usate = {n.id for n in ast.walk(ast.parse(SELEZIONE, mode="eval")) if isinstance(n, ast.Name)}
        righe += ["", "selection: " + SELEZIONE] + [
            "   %s = %s" % (k, e) for k, e in CONDIZIONI.items() if k in usate] + [
            "selected: %d of %d events (%.2f%%), %.3f Hz" % (SEL.sum(), N_EV, 100 * SEL.mean(),
                                                             SEL.sum() / T_TOT)]
    libero.text(0.0, 0.95, "\n".join(righe), va="top", fontsize=9, transform=libero.transAxes)
fig.tight_layout()
fig.savefig(os.path.join(USCITA, "carica_run_%s.png" % NOME_SEL), dpi=110)
plt.show()

In [ ]:
## === Confronto fra le run ===
# Ogni curva e' una run, o un gruppo di run consecutive se sono piu' di
# MAX_CURVE, divisa per il suo tempo di acquisizione: run di durata diversa
# si confrontano in rate, e una deriva del guadagno sposta le curve.
if len(NOMI) < 2:
    print("Una run sola: niente confronto.")
else:
    gruppi = np.array_split(np.arange(len(NOMI)), min(MAX_CURVE, len(NOMI)))
    _, lab, _ = etichette(NOMI)
    colori = plt.cm.viridis(np.linspace(0, 0.92, len(gruppi)))
    fig, assi, libero = pannelli("runs compared: rate per bin, each run divided by its acquisition time%s"
                                 % (" - selection: " + SELEZIONE if ATTIVA else ""))
    for a, c in zip(assi, CANALI):
        scelto = metodo(c)
        q = Q[c][scelto][SEL]
        lo, hi = intervallo(c, Q[c][scelto])
        bins = np.linspace(lo, hi, N_BIN + 1)
        for gr, col in zip(gruppi, colori):
            h, _ = np.histogram(q[np.isin(RUN_IDX[SEL], gr)], bins)
            a.stairs(h / DURATE[gr].sum(), bins, color=col, lw=1.0,
                     label=lab[gr[0]] if len(gr) == 1 else "%s - %s" % (lab[gr[0]], lab[gr[-1]]))
        a.set_title("ch%d %s - %s" % (c, RIVELATORE.get(c, ""), DESCR[scelto]), fontsize=9.5)
        a.set_yscale("log")
        a.set_xlim(lo, hi)
        a.set_xlabel("charge [ADC·ns]")
        a.set_ylabel("rate / %.0f ADC·ns [Hz]" % (bins[1] - bins[0]))
        a.grid(alpha=0.25)
    if libero is not None:
        libero.axis("off")
        libero.legend(*assi[0].get_legend_handles_labels(), loc="upper left", fontsize=9,
                      title="run" if all(len(gr) == 1 for gr in gruppi) else "runs (consecutive runs summed)",
                      frameon=False)
    else:
        assi[0].legend(fontsize=7.5, frameon=False)
    fig.tight_layout()
    fig.savefig(os.path.join(USCITA, "carica_run_%s_confronto.png" % NOME_SEL), dpi=110)
    plt.show()

In [ ]:
## === Salvataggio ===
# Le cariche in DATI, fuori da git: 50 run sono ~150 MB. In measurements/
# solo la descrizione, che dice anche dove sono.
os.makedirs(DATI, exist_ok=True)
npz = os.path.join(DATI, "carica_run_%s.npz" % NOME_SEL)
base = os.path.join(USCITA, "carica_run_" + NOME_SEL)
extra = {}
if all(len(x["r"]["ttt"]) == x["r"]["n"] for x in RISULTATI):
    extra["event_counter"] = np.concatenate([x["r"]["event_counter"] for x in RISULTATI])
    extra["trigger_time_tag"] = np.concatenate([x["r"]["ttt"] for x in RISULTATI])
meta = {
    "run": NOMI, "file": [x["path"] for x in RISULTATI],
    "eventi": [int(x["r"]["n"]) for x in RISULTATI],
    "durata_s": [float(d) for d in DURATE], "durata_da": [x["fonte"] for x in RISULTATI],
    "note": [x["note"] for x in RISULTATI],
    "saltate": SALTATE,
    "selezione": SELEZIONE, "condizioni": CONDIZIONI, "pe_adcns": {str(k): v for k, v in PE_ADCNS.items()},
    "selezionati": int(SEL.sum()),
    "dt_ns": DT, "unita": "ADC*ns, positiva nel verso del canale (1 pC = 204.8)",
    "canali": {str(c): {"rivelatore": RIVELATORE.get(c, ""), "verso": int(VERSI[c]),
                        "cancello_ns": [float(x) for x in FIN[c]["cancello_ns"]],
                        "cancello": FIN[c]["come"], "regola_cancello": regola(c),
                        "piedistallo_ns": [float(T_NS[FIN[c]["pi"][0]]), float(T_NS[FIN[c]["pi"][1] - 1])],
                        "piedistallo_scelto": metodo(c),
                        "regola_piedistallo": PIEDISTALLO[RIVELATORE[c]],
                        "picco_a_zero": {m: list(ZERO[c][m]) for m in ZERO[c]}}
               for c in CANALI},
    "eventi_per_blocco": EVENTI_PER_BLOCCO, "soglia_forma_conteggi": SOGLIA_FORMA,
    "campioni_finali_esclusi_dal_massimo": CAMPIONI_FINALI_ESCLUSI,
    "finestre_da": "primi %d eventi di %s" % (len(Y_RIF), NOMI_FILE[0]),
    "padme": "nessuna", "correzione_drs4": [x["drs4"] for x in RISULTATI],
    "creato": time.strftime("%Y-%m-%d %H:%M:%S"), "da": "tools/Distribuzioni_carica.ipynb",
    "cariche": npz,
}
np.savez_compressed(npz, run=np.array(NOMI), durata_s=DURATE, run_idx=RUN_IDX,
                    evento=EVENTO, **extra,
                    **{"q_evento_ch%d" % c: Q[c]["evento"] for c in CANALI},
                    **{"q_blocco_ch%d" % c: Q[c]["blocco"] for c in CANALI},
                    **{"saturo_ch%d" % c: SAT[c] for c in CANALI},
                    **{"ampiezza_%s_mV_ch%d" % (m, c): AMP[c][m] for c in CANALI for m in AMP[c]},
                    **{"t_max_ns_ch%d" % c: TMAX[c] for c in CANALI},
                    selezionato=SEL,
                    meta=json.dumps(meta))
with open(base + ".json", "w") as f:
    json.dump(meta, f, indent=1)
print("Cariche evento per evento: %s\nDescrizione e grafici:     %s.json, %s*.png" % (npz, base, base))

# Come funziona

Carica dei cinque canali del LNGS -- SiPM ch16-18, PMT ch20-21 -- dalle run della DAQ: una sola, o
tante, sommate e confrontate fra loro. Tutto quello che si sceglie sta nella cella **CONFIGURAZIONE**
in cima.

**Il metodo** e' quello di `tools/carica_padme.py`, lo strumento delle analisi di fine ottobre:

- **cancello e piedistallo automatici, per tipo di rivelatore** (`CANCELLO_AUTO`, `PIEDISTALLO`): la
  forma d'onda media degli eventi col massimo sopra `SOGLIA_FORMA` (PMT: mezzo PE, 73 conteggi; SiPM:
  100) dice dove l'impulso supera la
  `soglia` (10% del picco); il cancello va da `prima_ns` prima a `dopo_ns` dopo, o fino in fondo alla
  traccia meno `fine_traccia` campioni; il piedistallo da `inizio_ns` fino a `distacco_ns` prima del
  cancello. Con 10 e 20 ns e' la regola del monitor e di `carica_padme.py`, ed e' quella dei PMT. **Sui
  SiPM il cancello parte 60 ns prima della soglia e arriva in fondo alla traccia meno 20 campioni**
  (scelta dell'utente del 2026-11-03): con 10 ns la salita cominciava appena prima del cancello, e la
  regola del 10% lo chiudeva a ~680 ns con la coda ancora al 10-15% del picco. Cancelli e piedistalli
  si misurano **sulla prima run della lista** e valgono per tutte: cariche integrate in cancelli diversi
  non si sommano. Quelli scritti a mano in `CANCELLO` e `FINESTRA_PED` vincono;
- **le analisi della 0011 e della 0013 non usavano i cancelli automatici**: cancello 240-500 ns e
  piedistallo 0-150 ns, fissi per tutti i canali, ed e' con quelli che e' stato misurato
  K = 1647 ADC·ns/PE. Il cancello automatico dei PMT e' piu' stretto (~240-320 ns) e lascia fuori
  la luce lenta: per confrontarsi con quelle analisi c'e' una riga pronta nella configurazione;
- **carica = somma dei campioni nel cancello meno il piedistallo, per il passo di campionamento**,
  in ADC·ns (1 pC = 204.8 ADC·ns), col segno del verso del canale: positiva anche per impulsi
  negativi. Il verso lo dice il DCOffset scritto nel log della run (`VERSO` lo impone);
- **piedistallo per evento o per blocco.** Per evento e' la media della finestra in quell'evento,
  come nel monitor e in `carica_padme.py`. Per blocco e' la mediana di quelle medie su
  `EVENTI_PER_BLOCCO` eventi consecutivi (5000, ~90 s): segue le derive lente e non risente degli
  impulsi al buio che cadono nella finestra. **Di serie per blocco sui SiPM**: con un cancello lungo
  la sottrazione evento per evento allarga la carica (SiPM a 51 V: sigma 1500-2000 ADC·ns contro
  ~600 con la base fissa del canale, analisi del 2026-10-28), e un impulso al buio nella finestra
  porta la carica a valori negativi. Si calcolano tutte e due: il grafico mostra quella scelta e,
  in grigio, l'altra, con la larghezza del picco degli eventi senza impulso;
- **senza correzione PADME**, che sulla carica conta poco (stessa analisi): per quella c'e'
  `tools/carica_padme.py`;
- un evento con un campione al fondo scala nel cancello e' **saturo**: la sua carica e' un limite
  inferiore, e nel grafico e' a parte;
- **la selezione** (`CONDIZIONI`, `SELEZIONE`) si applica dopo la lettura, con le grandezze di ogni
  evento: ampiezza nel cancello in mV (A), carica in PE (Q, con `PE_ADCNS`) o in ADC·ns (C), istante
  del massimo (T), saturazione (S), col piedistallo scelto per il canale. Le espressioni ammettono
  confronti, and/or/not, parentesi, aritmetica e i contatori almeno/al_piu/esattamente. Cambiandola si
  riesegue la configurazione e poi dalla cella della selezione in giu', senza rileggere le run. Il file
  delle cariche tiene tutti gli eventi e la selezione come colonna (`selezionato`); grafici e file
  prendono nel nome la selezione. Sui canali usati nella selezione la distribuzione ha dentro il taglio.

**Dove gira.** Sul Mac col kernel **`Python (jupyter-global)`**, che legge le run dal bucket del
cloud montato in `~/nmv-data` (rclone, remoto `cygno-ro`): `~/nmv-data/WC/LNGS`, coi loro `.log`.
Sul PC DAQ col kernel `jupyter_env`, da `data/LNGS`. Il kernel deve girare sulla macchina dove sta
il notebook: uno di un altro server (il CNAF) non trova il repository.

**Dal bucket montato si legge a 1.4-4 MB/s per file**, ~7 MB/s con tre insieme (misurato il
2026-11-03): una run da 412 MB costa minuti, mentre decomprimerla e calcolarne le cariche costa ~5 s.
Per questo **ogni run si scarica una volta sola, nella copia locale** (`COPIA_LOCALE`, di serie
`~/nmv-copia/WC/LNGS`, col suo `.log`), con `SCARICHI_IN_PARALLELO` run scaricate in anticipo mentre
si leggono quelle arrivate; da li' si decomprime in un file temporaneo, si legge, e il temporaneo si
cancella. Rileggere una run con cancelli o piedistalli diversi costa quindi secondi, non minuti.
Una copia si tiene solo quando e' arrivata tutta, e si riscarica se sul bucket il file ha cambiato
dimensione (una run rimandata) o se non si decomprime piu'. Con meno di `SPAZIO_LIBERO_MIN_GB` liberi
sul disco le run nuove si leggono senza tenerne la copia. **Col bucket non montato** le run si cercano
nella copia: quelle gia' scaricate si analizzano anche senza rete. Tutte le run del LNGS sul cloud
pesano ~76 GB (283 run al 2026-11-04). Sul PC DAQ la copia non serve e non si fa: i file sono gia' sul
disco. In piu' le cariche di ogni run restano in cache (`data/carica/run/`), e rieseguendo con gli
stessi cancelli non si legge nemmeno la copia; anche gli eventi della prima run su cui si misurano i
cancelli restano in cache. Sul cloud un `.h5` non compresso non e' mai una run buona (sono copie prese a run in corso, o
di run chiuse male): si salta, e vale il `.h5.gz`.

**Cosa scrive.** I grafici e un `.json` che dice come sono state fatte le cariche in
`measurements/carica_run_<run>*`; le cariche evento per evento in `data/carica/carica_run_<run>.npz`,
fuori da git perche' pesano ~3 MB per run. Le run gia' lette restano in `data/carica/run/`:
rieseguendo con una lista piu' lunga si leggono solo quelle nuove. Per rileggere le cariche:

```python
z = np.load("data/carica/carica_run_....npz")
q = z["q_blocco_ch16"]; meta = json.loads(str(z["meta"]))
```